# בקרת איכות → רשימת מחיקות ל-Roboflow

**מה המחברת עושה:** סורקת את הדאטהסט פעם אחת ומייצרת קובץ `roboflow_delete_list.txt` —
רשימה מקובצת לפי סיבה, עם **שמות הקבצים כפי שהם ב-Roboflow** להדבקה ישירה בחיפוש שם.

**מה היא לא עושה:** לא מוחקת, לא משנה, ולא נוגעת ב-Roboflow. Roboflow הוא מקור האמת —
כל תיקון נעשה שם ידנית, ואז `Generate` גרסה חדשה.

| | |
|---|---|
| איפה רצה | מקומית, VS Code / Jupyter |
| מקום פנוי נדרש | ~1.1GB בשיא (zip + מחולץ). ה-zip נמחק מיד אחרי החילוץ. |
| פלט | `roboflow_delete_list.txt` — רשימת המחיקות |
| זמן ריצה | 1-3 דקות (ההורדה היא רוב הזמן) |

**סדר עבודה:** להריץ הכל → לפתוח את ה-txt → למחוק/לתקן ב-Roboflow → `Generate` גרסה חדשה →
**לשלוח לינק חדש לצוות** (הלינק הישן הוא snapshot ולא מתעדכן לבד) → להחליף `ROBOFLOW_URL` למטה ולהריץ שוב לאימות.

## §0 — הגדרות

כל מה שאולי תרצי לשנות נמצא כאן.

In [ ]:
from pathlib import Path

# לינק הייצוא מ-Roboflow (Download Dataset > YOLOv8 > show download code > Terminal)
# הלינק פג אחרי זמן מה — אם מקבלים 404, לייצא מחדש ולהדביק לינק טרי.
ROBOFLOW_URL = "https://app.roboflow.com/ds/lhrziicySt?key=K9FweSTnV6"

# כתובת הפרויקט עצמו — נכנסת לראש קובץ הפעולות כדי שיהיה לאן ללחוץ
PROJECT_URL  = "https://app.roboflow.com/chagitvain02-gmail-com/wheelchair-9qvfx-bchvo"

DATASET_DIR = Path("dataset")        # לאן לחלץ, יחסית לתיקיית המחברת
OUT_TXT     = Path("roboflow_delete_list.txt")

EXPECTED_CLASSES = ["person", "wheelchair", "people_wheelchair"]

NEAR_DUP_BITS = 5    # 0=זהות ויזואלית מוחלטת. 5 = "כמעט אותה תמונה". להעלות = יותר ממצאים.
MIN_SHORT_SIDE = 32  # תמונה שהצלע הקצרה שלה קטנה מזה - חשודה כזבל
BOX_EPS = 1e-3       # טולרנס לחריגת בוקס מ-0..1. 1e-3 ≈ 0.64px בתמונה 640 — עיגול של הייצוא, לא טעות תיוג.

print("יעד חילוץ:", DATASET_DIR.resolve())

## §1 — הורדה וחילוץ

בודק מקום פנוי לפני שמתחיל, מדלג אם הדאטהסט כבר קיים, ומוחק את ה-zip מיד אחרי החילוץ.

In [ ]:
import shutil, urllib.request, zipfile

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def has_images(root: Path) -> bool:
    return root.is_dir() and any(
        p.suffix.lower() in IMG_EXT for p in root.rglob("*") if p.is_file())

if has_images(DATASET_DIR):
    print("✅ הדאטהסט כבר קיים ב-", DATASET_DIR.resolve(), "— מדלג על ההורדה.")
    print("   (למשוך גרסה חדשה: למחוק את התיקייה ולהריץ שוב)")
else:
    free_gb = shutil.disk_usage(Path.cwd().anchor).free / 1024**3
    print(f"מקום פנוי: {free_gb:.1f}GB")
    if free_gb < 1.5:
        raise SystemExit(
            f"❌ רק {free_gb:.1f}GB פנויים, וצריך ~1.1GB בשיא.\n"
            "   פני מקום, או הריצי את המחברת ב-Colab (שם ההורדה לדיסק זמני של גוגל)."
        )

    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    zip_path = DATASET_DIR / "_roboflow.zip"

    def progress(block, block_size, total):
        done = block * block_size
        pct = f"{100 * done / total:5.1f}%" if total > 0 else "  ?  "
        print(f"\rמוריד... {pct}  ({done / 1024**2:6.1f}MB)", end="")

    print("מוריד מ-Roboflow (496MB, כמה דקות)...")
    urllib.request.urlretrieve(ROBOFLOW_URL, zip_path, reporthook=progress)
    print("\nמחלץ...")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATASET_DIR)
    zip_path.unlink()          # לא מחזיקים 496MB מיותרים
    print("✅ הדאטהסט ב-", DATASET_DIR.resolve(), " (ה-zip נמחק)")

## §2 — איתור ה-splits ובדיקת סדר הקלאסים

**למה סדר הקלאסים קריטי:** YOLO לא שומר שמות בקבצי התיוג, רק מספרים. שורת תיוג היא
`2 0.51 0.44 0.18 0.33` — וה-`2` מקבל משמעות רק מהמיקום שלו ברשימה ב-`data.yaml`.
אם הסדר מתחלף בין מה שאת שולחת למה שבקוד של מי שמאמנת, שום דבר לא ייכשל: האימון ירוץ,
המספרים ייראו סבירים, והמודל פשוט ילמד קלאסים הפוכים. בדיקה של שנייה כאן חוסכת יום.

In [ ]:
import yaml

ALIASES = {"train": "train", "valid": "valid", "val": "valid",
           "validation": "valid", "test": "test"}

splits = {}
for d in DATASET_DIR.rglob("*"):
    if not d.is_dir():
        continue
    # מבנה Roboflow: train/images + train/labels   |   מבנה חלופי: images/train + labels/train
    if d.name == "images" and (key := ALIASES.get(d.parent.name.lower())):
        splits[key] = {"images": d, "labels": d.parent / "labels"}
    elif d.parent.name == "images" and (key := ALIASES.get(d.name.lower())):
        splits[key] = {"images": d, "labels": d.parent.parent / "labels" / d.name}

if not splits:
    raise SystemExit("לא נמצאו תיקיות split תחת " + str(DATASET_DIR))

for k in ("train", "valid", "test"):
    if k in splits:
        n = sum(1 for p in splits[k]["images"].iterdir()
                if p.is_file() and p.suffix.lower() in IMG_EXT)
        print(f"  {k:6s}: {n:5d} תמונות   ({splits[k]['images'].relative_to(DATASET_DIR)})")
    else:
        print(f"  {k:6s}:     — לא קיים")

# ── סדר הקלאסים ──────────────────────────────────────────────────────────────
DATA_YAML = sorted(DATASET_DIR.rglob("data.yaml"))[0]
names = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))["names"]
if isinstance(names, dict):
    names = [names[k] for k in sorted(names, key=int)]
CLASS_NAMES = list(names)

print("\nסדר בפועל :", CLASS_NAMES)
print("סדר צפוי  :", EXPECTED_CLASSES)
CLASSES_OK = CLASS_NAMES == EXPECTED_CLASSES
print("✅ תואם" if CLASSES_OK
      else "🔴 לא תואם! כל מי שמאמנת חייבת להשתמש בסדר שב-data.yaml, לא בסדר שבראש.")

## §3 — סריקת כל התמונות (פעם אחת)

פותח כל קובץ פעם אחת ואוסף הכל לטבלה: האם נפתח, גודל, ושתי טביעות אצבע —

- **MD5** של הבייטים: שני קבצים עם אותו MD5 זהים לחלוטין. מוצא עותקים מדויקים, אבל עיוור
  לשינוי הקטן ביותר — שמירה מחדש ב-JPEG אחר משנה אותו לגמרי.
- **dHash**: מקטין ל-9×8 בגווני אפור ומקודד רק *האם* כל פיקסל בהיר מזה שמימינו → 64 ביט.
  שתי תמונות שנראות אותו דבר מקבלות hash כמעט זהה, גם אם הקבצים שונים. ההפרש נמדד בביטים.

In [ ]:
import hashlib
import numpy as np
import pandas as pd
from PIL import Image

def dhash(img, size=8) -> int:
    small = img.convert("L").resize((size + 1, size), Image.Resampling.LANCZOS)
    a = np.asarray(small, dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).flatten()       # 8x8 = 64 השוואות שמאל-ימין
    return int.from_bytes(np.packbits(bits.astype(np.uint8)).tobytes(), "big")

rows = []
for split, entry in splits.items():
    for p in sorted(entry["images"].iterdir()):
        if not (p.is_file() and p.suffix.lower() in IMG_EXT):
            continue
        rec = {"split": split, "file": p.name, "stem": p.stem, "path": str(p),
               "readable": True, "error": "", "w": 0, "h": 0, "md5": "", "dh": 0}
        try:
            with Image.open(p) as im:
                im.verify()                        # שלב 1: כותרת הקובץ
            with Image.open(p) as im:
                im.load()                          # שלב 2: פענוח מלא (תופס הורדה קטועה)
                rec["w"], rec["h"] = im.size
                rec["dh"] = dhash(im)
            rec["md5"] = hashlib.md5(p.read_bytes()).hexdigest()
        except Exception as e:
            rec["readable"], rec["error"] = False, f"{type(e).__name__}: {e}"
        rows.append(rec)

scan = pd.DataFrame(rows)
ok = scan[scan["readable"]].reset_index(drop=True)
print(f"נסרקו {len(scan)} תמונות · תקינות: {len(ok)} · פגומות: {len(scan) - len(ok)}")

## §4 — פגומות, כפילויות, רזולוציה

| מה נמצא | חומרה | למה |
|---|---|---|
| קובץ פגום | 🔴 | מפיל את האימון באמצע, לפעמים אחרי חצי שעה |
| כפילות **בין** splits | 🔴 | דליפת דאטה — המודל ראה את הוולידציה מראש, הציון מנופח |
| כפילות **בתוך** split | 🟡 | התמונה נספרת פעמיים, מטה מעט את האימון |
| כמעט-כפילות | 🟡/🔴 | אותו דבר, פשוט קשה יותר לגלות |

In [ ]:
from collections import defaultdict

# ── פגומות ───────────────────────────────────────────────────────────────────
corrupt = scan[~scan["readable"]]
print(f"קבצים פגומים: {len(corrupt)}")

# ── כפילויות מדויקות (MD5) ───────────────────────────────────────────────────
groups = defaultdict(list)
for r in ok.itertuples():
    groups[r.md5].append((r.split, r.file))
exact_dupes = {k: v for k, v in groups.items() if len(v) > 1}
cross_exact = {k: v for k, v in exact_dupes.items() if len({s for s, _ in v}) > 1}
print(f"כפילויות מדויקות: {len(exact_dupes)} קבוצות · מתוכן חוצות splits: {len(cross_exact)}")

# ── כמעט-כפילויות (dHash) ────────────────────────────────────────────────────
# מרחק Hamming בין כל זוג: a@(1-b) + (1-a)@b  -> מטריצה אחת קטנה במקום לולאה כפולה
bits = np.array([[int(c) for c in format(v, "064b")] for v in ok["dh"]], dtype=np.int16)
inv  = 1 - bits
dist = bits @ inv.T + inv @ bits.T
np.fill_diagonal(dist, 99)

near_pairs = []
for i, j in zip(*np.where(np.triu(dist <= NEAR_DUP_BITS, k=1))):
    a, b = ok.iloc[i], ok.iloc[j]
    if a["md5"] == b["md5"]:
        continue                                   # כבר נספר ככפילות מדויקת
    near_pairs.append((a["split"], a["file"], b["split"], b["file"],
                       int(dist[i, j]), a["split"] != b["split"]))
cross_near = [p for p in near_pairs if p[5]]
print(f"כמעט-כפילויות (≤{NEAR_DUP_BITS} ביט): {len(near_pairs)} זוגות · "
      f"מתוכן חוצות splits: {len(cross_near)}")

# ── רזולוציה ─────────────────────────────────────────────────────────────────
low_res = ok[ok[["w", "h"]].min(axis=1) < MIN_SHORT_SIDE]
print(f"תמונות עם צלע קצרה < {MIN_SHORT_SIDE}px: {len(low_res)}")

sizes = ok.groupby(["w", "h"]).size().sort_values(ascending=False)
print(f"\nגדלים נפוצים (מתוך {len(sizes)} גדלים שונים):")
print(sizes.head(3).to_string())

## §5 — תקינות התיוג

עוברים על כל `.txt` שורה-שורה. פורמט YOLO: `class_id x_center y_center width height`,
כשכל הקואורדינטות **מנורמלות ל-0..1** — ולכן אפשר לבדוק חריגה בלי לפתוח את התמונה.

**למה יש טולרנס (`BOX_EPS`):** הייצוא של Roboflow מעגל את הקואורדינטות, ולכן בוקס שנוגע
בדיוק בקצה התמונה יוצא לפעמים `1.0000078` במקום `1.0` — חריגה של חמש אלפיות פיקסל, שנראית
מושלמת בעין ואין מה לתקן בה. בלי טולרנס כל בוקס כזה היה נוחת ברשימת הפעולות כ"תיוג שבור".

| ממצא | מה עושים |
|---|---|
| בוקס חורג מ-0..1 ביותר מ-`BOX_EPS` / ברוחב או גובה 0 | 🔴 לתקן ב-Annotate |
| `class_id` מחוץ לטווח | 🔴 תיוג שבור |
| שורה שאינה 5 מספרים | 🔴 קובץ פגום |
| קובץ תיוג בלי תמונה (orphan) | 🔴 משהו נשבר בייצוא |
| תמונה בלי תיוג / תיוג ריק | 🟡 לבדוק — או שהיא רקע בכוונה, או שפספסו אותה |

In [ ]:
label_problems = []          # (split, image_file, kind, detail)
orphan_labels  = []
empty_labels   = []
instances      = []          # לספירת ההתפלגות ב-§6

def add(split, img, kind, detail=""):
    label_problems.append((split, img, kind, detail))

for split, entry in splits.items():
    # מ-scan ולא מ-ok: לתמונה פגומה יש קובץ תיוג תקין, והוא לא "יתום"
    img_by_stem = {r.stem: r.file for r in scan.itertuples() if r.split == split}
    lbl_dir = entry["labels"]
    lbl_stems = set()

    for lp in sorted(lbl_dir.glob("*.txt")) if lbl_dir.is_dir() else []:
        lbl_stems.add(lp.stem)
        img = img_by_stem.get(lp.stem)
        if img is None:
            orphan_labels.append((split, lp.name))
            continue

        lines = [ln for ln in lp.read_text(encoding="utf-8").splitlines() if ln.strip()]
        if not lines:
            empty_labels.append((split, img))
            continue

        for n, ln in enumerate(lines, 1):
            parts = ln.split()
            if len(parts) != 5:
                add(split, img, "שורת תיוג פגומה", f"שורה {n}: {len(parts)} ערכים במקום 5")
                continue
            try:
                cid = int(float(parts[0]))
                x, y, w, h = map(float, parts[1:])
            except ValueError:
                add(split, img, "שורת תיוג פגומה", f"שורה {n}: ערך שאינו מספר")
                continue

            if not 0 <= cid < len(CLASS_NAMES):
                add(split, img, "class_id מחוץ לטווח",
                    f"שורה {n}: class_id={cid}, קיימים 0..{len(CLASS_NAMES) - 1}")
                continue
            if w <= 0 or h <= 0:
                add(split, img, "בוקס בגודל אפס", f"שורה {n}: w={w}, h={h}")
            else:
                # החריגה הכי גדולה מבין ארבע הפאות. עד BOX_EPS זה עיגול של הייצוא
                # (תת-פיקסל, לא נראה לעין) ולא טעות תיוג — אין מה לתקן ב-Annotate.
                over = max(w / 2 - x, x + w / 2 - 1, h / 2 - y, y + h / 2 - 1)
                if over > BOX_EPS:
                    add(split, img, "בוקס חורג מגבולות התמונה",
                        f"שורה {n}: {CLASS_NAMES[cid]} ({x:.3f},{y:.3f},{w:.3f},{h:.3f}) "
                        f"— חריגה של {over:.3f} מהתמונה")
            instances.append({"split": split, "class": CLASS_NAMES[cid], "file": img})

    for stem, img in img_by_stem.items():
        if stem not in lbl_stems:
            empty_labels.append((split, img))

print(f"בעיות תיוג ממשיות : {len(label_problems)}")
print(f"תיוג יתום (בלי תמונה): {len(orphan_labels)}")
print(f"תמונות בלי תיוג/ריקות: {len(empty_labels)}")
if label_problems:
    display(pd.DataFrame(label_problems,
                         columns=["split", "תמונה", "בעיה", "פירוט"]).head(20))

## §6 — המספרים לדוח

**ההבחנה שחשוב לשמור:** "כמה תמונות מכילות את הקלאס" ≠ "כמה מופעים יש לקלאס".
תמונת רחוב אחת יכולה להכיל 15 בוקסים של `person`. המספר שקובע לאימון הוא **מופעים**.

In [ ]:
inst = pd.DataFrame(instances)
pivot = inst.pivot_table(index="class", columns="split", values="file",
                         aggfunc="count", fill_value=0)
for c in ("train", "valid", "test"):
    if c not in pivot.columns:
        pivot[c] = 0
pivot = pivot[["train", "valid", "test"]].reindex(CLASS_NAMES).fillna(0).astype(int)
pivot["מופעים"] = pivot.sum(axis=1)
pivot["תמונות"] = inst.groupby("class")["file"].nunique().reindex(CLASS_NAMES).fillna(0).astype(int)
display(pivot)

ratio = pivot["מופעים"].max() / max(pivot["מופעים"].min(), 1)
print(f"יחס חוסר-איזון בין הקלאס הנפוץ לנדיר: {ratio:.1f}:1", end="  ")
print("🟡 שווה להזכיר בדוח / לשקול class weights" if ratio > 3 else "✅ סביר")

## §7 — 🎯 קובץ המחיקות

**זה התוצר.** נשמר ל-`roboflow_delete_list.txt` בתיקיית המחברת, מקובץ לפי סיבה, עם שמות
הקבצים כפי ש-Roboflow מכיר אותם — בלי הסיומת `_jpg.rf.<hash>.jpg` שהייצוא מוסיף,
כי דווקא היא מה שמונע מהחיפוש למצוא אותם.

In [ ]:
import re
from collections import Counter

# ── שם הקובץ כפי ש-Roboflow מכיר אותו ────────────────────────────────────────
# הייצוא מוסיף לכל קובץ סיומת _jpg.rf.<hash>.jpg, ולכן חיפוש של השם המלא
# מהקבצים המקומיים לא מחזיר כלום ב-Roboflow. מחזירים את שם המקור.
_RF_SUFFIX = re.compile(r"\.rf\.[0-9A-Za-z]+\.\w+$")
_RF_EXT    = re.compile(r"_(jpg|jpeg|png|bmp|webp)$", re.I)

def _core(f):
    return _RF_EXT.sub("", _RF_SUFFIX.sub("", f))

_dup_names = Counter(_core(f) for f in scan["file"])

def rf(f):
    """שם לחיפוש ב-Roboflow. אם שני קבצים חולקים שם מקור - מוסיפים hash מבדיל."""
    c = _core(f)
    if _dup_names[c] > 1:
        return f"{c}   (עותק {f.split('.rf.')[-1].split('.')[0][:6]})"
    return c

L = []
A = L.append

A("=" * 72)
A("רשימת פעולות ל-ROBOFLOW")
A('נוצר אוטומטית ע"י dataset_qc.ipynb')
A("=" * 72)
A("")
A(f"הדאטהסט:  {PROJECT_URL}")
A("")
A("איך מוצאים תמונה:")
A("  Images (או Dataset)  ->  שדה החיפוש למעלה  ->  להדביק את השם מהרשימה.")
A("  אם החיפוש מבקש שאילתה:   filename:*ia_500000787*    (אפשר להוסיף  split:train)")
A("")
A("  השמות כאן הם שמות המקור כפי ש-Roboflow מכיר אותם, בלי הסיומת _jpg.rf.<hash>.jpg")
A("  שהייצוא מוסיף — ולכן חיפוש של השם המלא מהקבצים המקומיים לא מחזיר כלום.")
A("  אותו שם גם מוצא את הקובץ המקומי (חיפוש בתיקייה dataset).")
A("  שם קצר כמו 40 יחזיר הרבה תוצאות — לחפש אותו עם הסיומת:  40.jpg")
A('  "(עותק XXXXXX)" = שני קבצים שונים באותו שם מקור, להבחין ביניהם לפי התמונה עצמה.')
A("")
A("  אחרי כל המחיקות/התיקונים:  Generate  ->  גרסה חדשה  ->  לשלוח לינק חדש לצוות.")
A("")

def section(title, note=""):
    A("")
    A("-" * 72)
    A(title)
    if note:
        A("  " + note)
    A("-" * 72)

# ── [1] פגומים ───────────────────────────────────────────────────────────────
section(f"[1] 🔴 למחוק — קבצים פגומים ({len(corrupt)})",
        "הקובץ לא נפתח בכלל. יפיל את האימון באמצע.")
if len(corrupt) == 0:
    A("  ✅ אין.")
for r in corrupt.itertuples():
    A(f"  [{r.split}]  {rf(r.file)}")
    A(f"           סיבה: {r.error}")

# ── [2] כפילויות מדויקות ─────────────────────────────────────────────────────
section(f"[2] 🔴 למחוק — כפילויות מדויקות ({len(exact_dupes)} קבוצות)",
        "קבצים זהים bit-by-bit. להשאיר אחד, למחוק את השאר.")
if not exact_dupes:
    A("  ✅ אין.")
for i, v in enumerate(exact_dupes.values(), 1):
    cross = " ⚠ חוצה splits — דליפת דאטה!" if len({s for s, _ in v}) > 1 else ""
    A(f"  קבוצה {i}:{cross}")
    keep, *drop = sorted(v, key=lambda t: {"train": 0, "valid": 1, "test": 2}.get(t[0], 3))
    A(f"    ✔ להשאיר  [{keep[0]}]  {rf(keep[1])}")
    for s, f in drop:
        A(f"    ✘ למחוק   [{s}]  {rf(f)}")

# ── [3] כמעט-כפילויות ────────────────────────────────────────────────────────
section(f"[3] 🟡 לבדוק ולמחוק אחת מכל זוג — כמעט-כפילויות ({len(near_pairs)} זוגות)",
        "נראות אותו דבר אבל הקובץ שונה. זוג חוצה-splits הוא הבעיה החמורה.")
if not near_pairs:
    A("  ✅ אין.")
for s1, f1, s2, f2, d, cross in sorted(near_pairs, key=lambda p: (not p[5], p[4])):
    A(f"  {'⚠ חוצה splits' if cross else '  אותו split '} (הפרש {d} ביט)")
    A(f"    [{s1}]  {rf(f1)}")
    A(f"    [{s2}]  {rf(f2)}")

# ── [4] תיוג שבור ────────────────────────────────────────────────────────────
by_img = defaultdict(list)
for split, img, kind, detail in label_problems:
    by_img[(split, img)].append(f"{kind} — {detail}" if detail else kind)

section(f"[4] 🔴 לתקן ב-Annotate (לא למחוק) — תיוג שבור ({len(by_img)} תמונות)",
        "לפתוח את התמונה ב-Roboflow > Annotate ולתקן את הבוקס.")
if not by_img:
    A("  ✅ אין.")
for (split, img), probs in sorted(by_img.items()):
    A(f"  [{split}]  {rf(img)}")
    for p in probs:
        A(f"           {p}")

# ── [5] תיוג יתום ────────────────────────────────────────────────────────────
section(f"[5] 🔴 תיוג בלי תמונה ({len(orphan_labels)})",
        "קובץ .txt שאין לו תמונה. סימן שהייצוא נשבר — לייצא מחדש.")
if not orphan_labels:
    A("  ✅ אין.")
for s, f in orphan_labels:
    A(f"  [{s}]  {rf(f)}")

# ── [6] בלי תיוג ─────────────────────────────────────────────────────────────
section(f"[6] 🟡 לבדוק — תמונות בלי תיוג ({len(empty_labels)})",
        "או שהן רקע בכוונה, או שפספסו אותן. אם פספסו — המודל לומד במפורש 'כאן אין כלום'.")
if not empty_labels:
    A("  ✅ אין.")
for s, f in sorted(empty_labels)[:200]:
    A(f"  [{s}]  {rf(f)}")
if len(empty_labels) > 200:
    A(f"  ... ועוד {len(empty_labels) - 200}. אם המספר גדול — כנראה שזו התנהגות מכוונת של הייצוא.")

# ── [7] רזולוציה נמוכה ───────────────────────────────────────────────────────
section(f"[7] 🟡 לבדוק — רזולוציה נמוכה ({len(low_res)})",
        f"צלע קצרה < {MIN_SHORT_SIDE}px. בדרך כלל זבל שנכנס בטעות.")
if len(low_res) == 0:
    A("  ✅ אין.")
for r in low_res.itertuples():
    A(f"  [{r.split}]  {rf(r.file)}   ({r.w}x{r.h})")

# ── סיכום ────────────────────────────────────────────────────────────────────
n_delete = (len(corrupt)
            + sum(len(v) - 1 for v in exact_dupes.values()))
A("")
A("=" * 72)
A("סיכום")
A("=" * 72)
A(f"  למחוק בוודאות        : {n_delete}")
A(f"  לבדוק ולהחליט        : {len(near_pairs)} זוגות + {len(low_res)} רזולוציה + {len(empty_labels)} בלי תיוג")
A(f"  לתקן ב-Annotate      : {len(by_img)}")
A(f"  סדר הקלאסים          : {'✅ תקין' if CLASSES_OK else '🔴 לא תואם לצפוי — לתאם עם הצוות'}")
A(f"  סה\"כ תמונות בדאטהסט  : {len(scan)}")
A("")
A("אחרי הביצוע:  Generate גרסה חדשה  ->  להחליף ROBOFLOW_URL ב-§0  ->  להריץ שוב לאימות.")

OUT_TXT.write_text("\n".join(L), encoding="utf-8")
print(f"✅ נשמר: {OUT_TXT.resolve()}")
print()
print("\n".join(L[-12:]))